Requirements:
- python 3.11.5
- selenium 4.16.0

In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from IPython.display import display, HTML

# due to bot detection, we need to use this library
import undetected_chromedriver as uc

import time
import json

In [ ]:
# with open("login-lugoblogger.json", 'r') as fp:
#   data = json.load(fp)

with open("login.json", 'r') as fp:
  data = json.load(fp)


In [45]:
def get_default_chrome_options():
  options = webdriver.ChromeOptions()
  options.add_argument("--no-sandbox")
  return options

In [ ]:
# PATH = "/usr/bin/chromedriver"
# cService = webdriver.ChromeService(executable_path=PATH)
# driver = webdriver.Chrome(service=cService)
# driver = webdriver.Chrome()
# driver = webdriver.Firefox()

# options = get_default_chrome_options()
# driver = webdriver.Chrome(options=options)

# Configure Chrome options to reduce bot detection
# chrome_options = Options()
# chrome_options.add_argument("--disable-blink-features=AutomationControlled")
# chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
# chrome_options.add_experimental_option('useAutomationExtension', False)
# chrome_options.add_argument("--disable-web-security")
# chrome_options.add_argument("--disable-features=WebRTC")
# chrome_options.add_argument("--disable-extensions")
# chrome_options.add_argument("--no-sandbox")
# chrome_options.add_argument("--disable-dev-shm-usage")

# driver = webdriver.Chrome(options=chrome_options)

# # Remove webdriver property
# driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

In [58]:
driver = uc.Chrome(headless=False,use_subprocess=False)

In [59]:
target_url = "https://twitter.com/scrapingdog"
# login_url = "https://x.com/i/flow/login"

driver.get(target_url)
# driver.get(login_url)
time.sleep(5)

# resp = driver.page_source
# driver.close()
# print(resp)

Click "Log in"

In [60]:
# element = driver.find_elements(By.XPATH, f"//span[text()='Log in']")
# element = driver.find_elements(By.XPATH, f"//span[text()='Masuk']")
element = driver.find_elements(By.XPATH, f"//span[text()='ログイン']")
element[0].click()

Put your "Username".
Run this many times if it failed

In [64]:
element = driver.find_element(By.XPATH, f"//input[@autocomplete='username']")
element.clear()
element.send_keys(data["email"])
element.send_keys(Keys.RETURN)

time.sleep(2)

Asking "username" or "phone number" because of unintended activity

In [65]:
element = driver.find_element(By.XPATH, f"//input[@autocomplete='on']")
element.clear()
element.send_keys(data["username"])
element.send_keys(Keys.RETURN)

time.sleep(2)

In [66]:
# element = driver.find_element(By.XPATH, f"//input[@name='text']")
element = driver.find_element(By.XPATH, f"//input[@name='password']")
element.clear()
element.send_keys(data["pass"])
element.send_keys(Keys.RETURN)

In [ ]:
driver

In [ ]:
resp = driver.page_source
# driver.close()
print(resp)

Extracting Profile name, handle, bio, category, website, following, followers

In [ ]:
soup = BeautifulSoup(resp, "html.parser")
profile_header = soup.find("div", {"data-test": "UserProfileHeader_Items"})
soup.find("div", {"class": "r-1vr29t4"}).text

In [ ]:
o = {}
soup = BeautifulSoup(resp, "html.parser")

try:
  o["profile_name"] = soup.find("div", {"class": "r-1vr29t4"}).text
except:
  o["profile_name"] = None
  
try:
  o["profile_handle"] = soup.find("div", {"class": "r-1wvb978"}).text
except:
  o["profile_handle"] = None
  
try:
  o["profile_bio"] = soup.find("div", {"data-testid": "UserDescription"}).text
except:
  o["profile_bio"] = None
  
profile_header = soup.find("div", {"data-testid": "UserProfileHeader_Items"})
try:
  o["profile_category"] = profile_header.find(
    "span", {"data-testid": "UserProfessionalCategory"}).text
except:
  o["profile_category"] = None
  
try:
  o["profile_website"] = profile_header.find("a").get("href")
except:
  o["profile_website"] = None
  
try:
  o["profile_joining_date"] = profile_header.find(
    "span", {"data-testid": "UserJoinDate"}).text
except:
  o["profile_joining_date"] = None
  
try:
  o["profile_following"] = soup.find_all("a", {"class": "r-rjixqe"})[1].text
except:
  o["profile_following"] = None
  
try:
  o["profile_followers"] = soup.find_all("a", {"class": "r-rjixqe"})[2].text
except:
  o["profile_followers"] = None


for k, v in o.items():
  print(f"{k:>20s}:", v)